In [1]:
# @title Setup Constants

from google.colab import drive
drive.mount('/content/drive')

import numpy as np

np.random.seed(42)

n_ad_genre = 10

ad_genre = '''Airlines
Apparel
Automotive
Electronics
Fast-Moving Consumer Goods (FMCG)
Finance
Hotels
Media
Packaged Food (and Beverage)
Restaurants'''

ad_genre_list = ad_genre.split("\n")

user_prompt = ""
organic_response = ""

n_slots = 20

In [2]:
# @title Generate Bids and Preference
def generate_bids_and_preference(n_bidders: int, n_genres: int):
    """
    Generates random floating point bids following a normal distribution clipped to [0, 1] and binary preferences with a Poisson distribution
    for the number of preferred genres per bidder.

    Args:
        n_bidders: The number of bidders.
        n_genres: The number of ad genres.

    Returns:
        A tuple containing:
            - bids: A numpy array of random float bids.
            - preferences: A numpy array of random binary preferences.
    """
    # Generate bids from a normal distribution, and clip to [0, 1]
    bids = np.random.normal(loc=0.5, scale=0.2, size=n_bidders)
    bids = np.clip(bids, 0, 1)

    preferences = np.zeros((n_bidders, n_genres), dtype=int)

    for i in range(n_bidders):
        # Generate number of preferred genres using Poisson distribution, ensuring at least 1
        num_preferred_genres = np.random.poisson(lam=2)
        num_preferred_genres = min(max(1, num_preferred_genres),n_genres)

        # Randomly select preferred genres
        preferred_genres_indices = np.random.choice(n_genres, size=num_preferred_genres, replace=False)
        preferences[i, preferred_genres_indices] = 1

    return bids, preferences


bids, preferences = generate_bids_and_preference(100, n_ad_genre)
print(bids.shape, preferences.shape)

(100,) (100, 10)


In [3]:
# @title Generate Coherence Matrix
def generate_coherence_matrix(n_slots, n_ad_genre, dummy_coherence=True, user_prompt=None, ad_genre_list=None, organic_response=None):
    """
    Generates a coherence matrix between ad slots and genres.

    Args:
        n_slots: The number of ad slots.
        n_ad_genre: The number of ad genres.
        dummy_coherence: If True, generates a dummy coherence matrix with random values.
                         If False, the coherence matrix is initialized with zeros and can be
                         updated based on user_prompt and organic_response.
        ad_genre_list: A list of ad genres.
        user_prompt: The user's search query (not used in dummy generation).
        organic_response: The organic search result (not used in dummy generation).

    Returns:
        A numpy array representing the coherence matrix.
    """
    coherence_matrix = np.zeros((n_slots, n_ad_genre))
    if dummy_coherence:
        # Generate dummy coherence values as floats between 0 and 1
        coherence_matrix = np.random.rand(n_slots, n_ad_genre)

        # Set some values to a very low number to represent low coherence
        low_coherence_indices = np.random.choice(n_slots * n_ad_genre, size=int(0.5 * n_slots * n_ad_genre), replace=False)
        coherence_matrix.flat[low_coherence_indices] = -1000000000

        # Ensure each row has at least one value not equal to -1000000000
        for i in range(n_slots):
            if np.all(coherence_matrix[i, :] == -1000000000):
                random_genre_index = np.random.randint(n_ad_genre)
                coherence_matrix[i, random_genre_index] = np.random.rand()

    else:
        pass

    return coherence_matrix

coherence_matrix = generate_coherence_matrix(n_slots, n_ad_genre)
print(coherence_matrix.shape)

(20, 10)


In [4]:
# @title Compute Matching Matrix
def compute_matching_matrix(bids, preferences, coherence_matrix, alpha = 1):
    """
    Computes the matching matrix based on bids, preferences, and coherence matrix.

    Args:
        bids: A numpy array of bids, shape = (n_bidders, ).
        preferences: A numpy array of preferences over each genre, shape = (n_bidders, n_genres).
        coherence_matrix: A numpy array representing the coherence matrix, shape = (n_slots, n_genres).
        alpha: The weight for the coherence matrix.

    Returns:
        A numpy array representing the matching matrix, shape = (n_bidders, n_slots).
    """
    n_bidders, n_genres = preferences.shape
    n_slots, _ = coherence_matrix.shape
    matching_matrix = np.ones((n_bidders, n_slots)) * -1000000000

    for i in range(n_bidders):
        for j in range(n_slots):
            for k in range(n_genres):
                if preferences[i, k] == 1:
                    matching_matrix[i, j] = max(matching_matrix[i, j], alpha * coherence_matrix[j, k] + bids[i])
                else:
                    matching_matrix[i, j] = max(matching_matrix[i, j], alpha * coherence_matrix[j, k])

    return matching_matrix

compute_matching_matrix(bids, preferences, coherence_matrix)

array([[1.44389223, 0.82891547, 0.87009887, ..., 0.97348897, 0.77040742,
        1.27984213],
       [1.41661363, 0.82891547, 0.87009887, ..., 0.97348897, 0.96099033,
        1.15284644],
       [0.94426649, 0.82891547, 0.94829021, ..., 0.97542076, 0.77040742,
        0.79480955],
       ...,
       [1.49647754, 0.82891547, 0.87009887, ..., 0.97348897, 1.04085424,
        0.79480955],
       [0.94426649, 0.82891547, 0.87009887, ..., 0.97348897, 0.77040742,
        0.79480955],
       [1.39734906, 0.82891547, 1.07897395, ..., 1.42657154, 0.94172576,
        1.24789212]])

In [5]:
# @title VCG Mechanism with MinCostFlow

import heapq

class MinCostFlow:
    def __init__(self, n):
        self.n = n
        self.adj = [[] for _ in range(n)]

    def add_edge(self, u, v, cap, cost):
        # forward edge
        self.adj[u].append([v, cap, cost, len(self.adj[v])])
        # backward edge
        self.adj[v].append([u, 0, -cost, len(self.adj[u]) - 1])

    def min_cost_flow(self, s, t, max_f):
        n = self.n
        flow, cost = 0, 0
        potential = [0]*n  # Johnson potentials
        INF = 10**18

        while flow < max_f:
            dist = [INF]*n
            parent = [(-1, -1)]*n
            dist[s] = 0
            pq = [(0, s)]
            while pq:
                d,u = heapq.heappop(pq)
                if d != dist[u]:
                    continue
                for i,(v,cap,w,rev) in enumerate(self.adj[u]):
                    if cap <= 0:
                        continue
                    nd = d + w + potential[u] - potential[v]
                    if nd < dist[v]:
                        dist[v] = nd
                        parent[v] = (u,i)
                        heapq.heappush(pq,(nd,v))
            if dist[t] == INF:
                return None  # cannot send more flow

            for v in range(n):
                if dist[v] < INF:
                    potential[v] += dist[v]

            # push 1 unit (all capacities are 1 in our matching reduction)
            add = max_f - flow
            v = t
            while v != s:
                u,i = parent[v]
                add = min(add, self.adj[u][i][1])
                v = u
            v = t
            while v != s:
                u,i = parent[v]
                e = self.adj[u][i]
                re = self.adj[v][e[3]]
                e[1] -= add
                re[1] += add
                cost += e[2]*add
                v = u
            flow += add
        return cost

def max_weight_exact_k(cost_matrix, K):
    """
    cost_matrix: list of lists of edge weights w_ij (float/int). Missing edges can be represented by None.
    Returns: (total_weight, matching) where matching is list of (i,j).
    Maximizes sum of selected weights subject to exactly K matches.
    """
    m = len(cost_matrix)
    n = max((len(row) for row in cost_matrix), default=0)
    if K < 0 or K > min(m, n):
        return None

    # Build bipartite network: S -> left(i) -> right(j) -> T, all caps 1.
    # Convert to *min-cost* by shifting: c_ij = Cmax - w_ij >= 0
    valid_edges = []
    max_w = None
    for i,row in enumerate(cost_matrix):
        for j,w in enumerate(row):
            if w is not None:
                max_w = w if max_w is None else max(max_w, w)
                valid_edges.append((i,j,w))
    if K == 0:
        return (0, [])

    if max_w is None:
        return None  # no edges at all

    Cmax = max_w
    # graph indices:
    # S=0, left 1..m, right m+1..m+n, T=m+n+1
    S = 0
    L0 = 1
    R0 = 1 + m
    T = 1 + m + n
    g = MinCostFlow(T+1)

    for i in range(m):
        g.add_edge(S, L0+i, 1, 0)
    for j in range(n):
        g.add_edge(R0+j, T, 1, 0)
    # edges i->j with cost = Cmax - w
    for i,j,w in valid_edges:
        g.add_edge(L0+i, R0+j, 1, Cmax - w)

    res_cost = g.min_cost_flow(S, T, K)
    if res_cost is None:
        return None  # can't achieve K matches

    # Recover matching by inspecting saturated edges L->R
    matching = []
    for i in range(m):
        for v,cap,cst,rev in g.adj[L0+i]:
            # originally capacity 1; if now 0, it was used
            # but we must ensure it's an L->R edge:
            if R0 <= v < R0+n and cap == 0:
                j = v - R0
                matching.append((i, j))
    # Compute true total weight
    total_weight = sum(cost_matrix[i][j] for i,j in matching)
    if len(matching) != K:
        # Extremely defensive: in principle shouldn't happen
        return None
    return (total_weight, matching)


def vcg_assignment_via_min_cost_flow(V, K):
    """
    VCG (Clarke pivot) payments for one-to-one assignment with quasilinear utilities.

    Args:
      V: (n_bidders x n_items) valuations; no -inf entries (all pairs allowed).
         Unmatched = outside option 0.
      K: exactly assign K items.

    Returns:
      matches: list of (bidder, item) using original indices
      payments: dict bidder -> payment
      welfare: total welfare of allocation
    """
    V = np.asarray(V, dtype=float)
    n_bidders, n_items = V.shape

    # --- 1) Welfare-maximizing allocation with all bidders
    welfare, match = max_weight_exact_k(V.tolist(), K)
    if welfare is None:
        raise ValueError("No feasible matching of size K")

    matches = [None]*n_bidders
    for i,j in match:
        matches[i] = j

    # --- 2) Compute VCG payments
    payments = {}
    for i in range(n_bidders):
        if matches[i] is None:
            payments[i] = 0.0
            continue

        # Welfare with bidder i removed
        mask = [row[:] for idx,row in enumerate(V.tolist()) if idx != i]
        res = max_weight_exact_k(mask, K)
        if res is None:
            W_minus_i = -np.inf  # infeasible if we can't assign K without i
        else:
            W_minus_i, _ = res

        v_i = V[i, matches[i]]
        payments[i] = W_minus_i - (welfare - v_i)

    return matches, payments, welfare


In [6]:
# @title VCG Mechanism with JV

from scipy.optimize import linear_sum_assignment

def vcg_assignment(V, K):
    """
    VCG (Clarke pivot) payments for one-to-one assignment.

    Args:
      V: (n_bidders x n_items) valuations; no -inf entries (all pairs allowed).
         Unmatched = outside option 0.
      K: exactly assign K items (must satisfy 0 <= K <= min(n_bidders, n_items)).

    Returns:
      matches: list of length n_bidders with item index or None (if unmatched)
      payments: list of length n_bidders with VCG payment for each bidder
      welfare: total welfare of allocation (sum of assigned bidders' valuations)
    """
    V = np.asarray(V, dtype=float)
    n_bidders, n_items = V.shape

    def solve_exact_k(V, K):
        """
        Solve welfare-maximizing assignment selecting exactly K real items,
        """
        n_bidders, n_items = V.shape

        if min(n_bidders,n_items) > K:
            V_pad = np.concat([V, np.ones((min(n_bidders,n_items) - K, n_items)) * 1000000000], axis = 0)
        else:
            V_pad = V

        row_ind, col_ind = linear_sum_assignment(-V_pad)

        # Extract matches for REAL bidders to REAL items only
        matches = [None] * n_bidders
        welfare = 0.0
        for r, c in zip(row_ind, col_ind):
            if r < n_bidders and c < n_items:
                matches[r] = c
                welfare += V[r, c]
        return matches, welfare

    # --- 1) Welfare-maximizing allocation with all bidders (exactly K items)
    matches, welfare = solve_exact_k(V, K)

    # --- 2) Compute VCG payments (Clarke pivot)
    payments = [0.0] * n_bidders
    # Value each bidder gets in the chosen allocation (0 if unmatched)
    bidder_value = [0.0] * n_bidders
    for i, j in enumerate(matches):
        if j is not None:
            bidder_value[i] = V[i, j]

    # Total welfare with everyone:
    W = welfare

    for i in range(n_bidders):
        # Remove bidder i and recompute optimal welfare for others
        V_minus_i = V.copy()
        V_minus_i[i,:] -= 1000000000
        _, W_minus_i = solve_exact_k(V_minus_i, K)

        # Clarke pivot: payment = (welfare of others without i) - (welfare of others with i)
        payments[i] = W_minus_i - (W - bidder_value[i])

    return matches, payments, welfare

# vcg_assignment(V=compute_matching_matrix(bids, preferences, coherence_matrix), K=10)

In [7]:
# @title Test VCG

V = np.array([
    [10,  5,  6],
    [ 9,  7,  8],
    [ 4, 11,  3],
    [ 1,  2,  3],
    [ 1,  2,  3],
    [ 1,  2,  3],
    [ 1,  2,  3],
    [ 1,  2,  3],
    [ 1,  2,  3],
])
matches, payments, welfare = vcg_assignment(V, K=2)
print("Matches:", matches)
print("Payments:", payments)
print("Welfare:", welfare)


matches, payments, welfare = vcg_assignment_via_min_cost_flow(V, K=2)
print("Matches:", matches)
print("Payments:", payments)
print("Welfare:", welfare)

Matches: [np.int64(0), None, np.int64(1), None, None, None, None, None, None]
Payments: [np.float64(9.0), np.float64(0.0), np.float64(8.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
Welfare: 21.0
Matches: [0, None, 1, None, None, None, None, None, None]
Payments: {0: np.float64(9.0), 1: 0.0, 2: np.float64(8.0), 3: 0.0, 4: 0.0, 5: 0.0, 6: 0.0, 7: 0.0, 8: 0.0}
Welfare: 21.0


In [8]:
# @title Greedy Mechanism

def greedy_assignment(V, K):
    """
    VCG (Clarke pivot) payments for one-to-one assignment.

    Args:
      V: (n_bidders x n_items) valuations; no -inf entries (all pairs allowed).
         Unmatched = outside option 0.
      K: exactly assign K items (must satisfy 0 <= K <= min(n_bidders, n_items)).

    Returns:
      matches: list of length n_bidders with item index or None (if unmatched)
      payments: list of length n_bidders with myerson payment for each bidder
      welfare: total welfare of allocation (sum of assigned bidders' valuations)
    """
    V = np.asarray(V, dtype=float)
    n_bidders, n_items = V.shape

    # Sort all bidder-item pairs by valuation in descending order
    # Create a list of (valuation, bidder_index, item_index) tuples
    all_pairs = []
    for i in range(n_bidders):
        for j in range(n_items):
            all_pairs.append((V[i, j], i, j))

    all_pairs.sort(key=lambda x: x[0], reverse=True)

    matches = [None] * n_bidders
    assigned_items = set()
    assigned_bidders = set()
    welfare = 0.0
    payments = [0.0] * n_bidders

    for i in range(len(all_pairs)):
        value, bidder_index, item_index = all_pairs[i]
        if item_index not in assigned_items and bidder_index not in assigned_bidders:

            temp_assigned_items = assigned_items.copy()
            temp_assigned_bidders = assigned_bidders.copy()
            payment_i = 0
            for j in range(i+1, len(all_pairs)):
                other_bidder_value, other_bidder_index, other_item_index = all_pairs[j]
                if other_bidder_index in temp_assigned_bidders or other_item_index in temp_assigned_items:
                    continue
                temp_assigned_items.add(other_item_index)
                temp_assigned_bidders.add(other_bidder_index)
                if len(temp_assigned_items) >= K:
                    payment_i = other_bidder_value
                    break

            payments[bidder_index] = payment_i

            matches[bidder_index] = item_index
            assigned_items.add(item_index)
            assigned_bidders.add(bidder_index)
            welfare += value

            if len(assigned_items) >= K:
                break

    # Compute Myerson payments

    return matches, payments, welfare

In [33]:
# @title Running

import time

n_bidders = 100
n_genres = 100
n_slots = 30
K = 5

num_runs = 100
vcg_welfares = []
greedy_welfares = []
vcg_runtimes = []
vcg_min_cost_flow_runtimes = []
greedy_runtimes = []

for t in range(num_runs):
    print(f"Run {t+1}/{num_runs}")
    # Generate new data for each run
    bids, preferences = generate_bids_and_preference(n_bidders, n_genres)
    coherence_matrix = generate_coherence_matrix(n_slots, n_genres)
    V = compute_matching_matrix(bids, preferences, coherence_matrix)

    # Measure VCG runtime and welfare
    start_time = time.time()
    vcg_matches, vcg_payments, vcg_welfare = vcg_assignment(V=V, K=K)
    end_time = time.time()
    vcg_runtimes.append(end_time - start_time)
    vcg_welfares.append(vcg_welfare)
    print(f"  VCG (Jonker-Volgenant) Runtime: {end_time - start_time:.6f} seconds")


    # Measure VCG min cost flow runtime
    start_time = time.time()
    vcg_min_cost_flow_matches, vcg_min_cost_flow_payments, vcg_min_cost_flow_welfare = vcg_assignment_via_min_cost_flow(V=V, K=K)
    end_time = time.time()
    vcg_min_cost_flow_runtimes.append(end_time - start_time)
    print(f"  VCG (min cost flow) Runtime: {end_time - start_time:.6f} seconds")

    if abs(vcg_min_cost_flow_welfare - vcg_welfare) > 0.000001:
        print(f"  VCG (Jonker-Volgenant) welfare: {vcg_welfare}")
        print(f"  VCG (min cost flow) welfare: {vcg_min_cost_flow_welfare}")
        raise ValueError("VCG (min cost flow) welfare does not match VCG (Jonker-Volgenant)")


    # Measure Greedy runtime and welfare
    start_time = time.time()
    greedy_matches, greedy_payments, greedy_welfare = greedy_assignment(V=V, K=K)
    end_time = time.time()
    greedy_runtimes.append(end_time - start_time)
    greedy_welfares.append(greedy_welfare)
    print(f"  Greedy Runtime: {end_time - start_time:.6f} seconds")


print(f"\nAverage VCG Runtime: {np.mean(vcg_runtimes):.6f} seconds")
print(f"Average VCG Min Cost Flow Runtime: {np.mean(vcg_min_cost_flow_runtimes):.6f} seconds")
print(f"Average Greedy Runtime: {np.mean(greedy_runtimes):.6f} seconds")
print(f"Average VCG Welfare: {np.mean(vcg_welfares):.2f}")
print(f"Average Greedy Welfare: {np.mean(greedy_welfares):.2f}")
print(f"Average Welfare Ratio (Greedy/VCG): {np.mean(greedy_welfares) / np.mean(vcg_welfares):.4f}")

Run 1/100
  VCG (Jonker-Volgenant) Runtime: 0.014376 seconds
  VCG (min cost flow) Runtime: 0.142831 seconds
  Greedy Runtime: 0.001978 seconds
Run 2/100
  VCG (Jonker-Volgenant) Runtime: 0.014149 seconds
  VCG (min cost flow) Runtime: 0.048269 seconds
  Greedy Runtime: 0.001848 seconds
Run 3/100
  VCG (Jonker-Volgenant) Runtime: 0.014605 seconds
  VCG (min cost flow) Runtime: 0.047985 seconds
  Greedy Runtime: 0.001861 seconds
Run 4/100
  VCG (Jonker-Volgenant) Runtime: 0.014148 seconds
  VCG (min cost flow) Runtime: 0.141753 seconds
  Greedy Runtime: 0.002029 seconds
Run 5/100
  VCG (Jonker-Volgenant) Runtime: 0.013913 seconds
  VCG (min cost flow) Runtime: 0.065554 seconds
  Greedy Runtime: 0.001868 seconds
Run 6/100
  VCG (Jonker-Volgenant) Runtime: 0.014766 seconds
  VCG (min cost flow) Runtime: 0.048283 seconds
  Greedy Runtime: 0.001879 seconds
Run 7/100
  VCG (Jonker-Volgenant) Runtime: 0.014891 seconds
  VCG (min cost flow) Runtime: 0.134770 seconds
  Greedy Runtime: 0.001890 

In [34]:
# @title Save the results
import os

working_dir = "/content/drive/MyDrive/LLM Auction-release/prototype_results/"

import pandas as pd

# Create a dictionary with the results
results_data = {
    'VCG_Jonker_Volgenant_Runtime': vcg_runtimes,
    'VCG_Min_Cost_Flow_Runtime': vcg_min_cost_flow_runtimes,
    'Greedy_Runtime': greedy_runtimes,
    'VCG_Welfare': vcg_welfares,
    'Greedy_Welfare': greedy_welfares
}

# Create a pandas DataFrame
results_df = pd.DataFrame(results_data)

# Save the DataFrame to a CSV file
results_df.to_csv(working_dir + f'{n_bidders}_bidders_{n_genres}_genres_{n_slots}_slots_{K}_insertions_auction_results.csv', index=False)

In [18]:
from scipy import stats

def mean_ci_t(x, alpha=0.05):
    """Two-sided (1-alpha) CI for the mean via Student-t (assumes IID)."""
    x = np.asarray(x, dtype=float)
    n = x.size
    m = np.mean(x)
    s = np.std(x, ddof=1)
    tcrit = stats.t.ppf(1 - alpha/2, df=n-1)
    halfwidth = tcrit * s / np.sqrt(n)
    return m, (m - halfwidth, m + halfwidth)

m_vcg_rt, ci_vcg_rt = mean_ci_t(vcg_runtimes)
m_vcg_mcf_rt, ci_vcg_mcf_rt = mean_ci_t(vcg_min_cost_flow_runtimes)
m_greedy_rt, ci_greedy_rt = mean_ci_t(greedy_runtimes)

print(f"Average VCG (Jonker-Volgenant) Runtime:  {m_vcg_rt:.6f} s  (95% CI: {ci_vcg_rt[0]:.6f}, {ci_vcg_rt[1]:.6f})")
print(f"Average VCG (min cost flow) Runtime:  {m_vcg_mcf_rt:.6f} s  (95% CI: {ci_vcg_mcf_rt[0]:.6f}, {ci_vcg_mcf_rt[1]:.6f})")
print(f"Average Greedy Runtime: {m_greedy_rt:.6f} s  (95% CI: {ci_greedy_rt[0]:.6f}, {ci_greedy_rt[1]:.6f})")

alpha = 0.05
ratios = np.array(greedy_welfares) / np.array(vcg_welfares)
lower = np.nanpercentile(ratios, 100 * (alpha/2))
upper = np.nanpercentile(ratios, 100 * (1 - alpha/2))
print(f"Welfare Ratio (Greedy/VCG): {np.mean(ratios):.4f}  (95% CI: {lower:.4f}, {upper:.4f})")


Average VCG (Jonker-Volgenant) Runtime:  1.128448 s  (95% CI: 1.092190, 1.164706)
Average VCG (min cost flow) Runtime:  1.536649 s  (95% CI: 1.494860, 1.578437)
Average Greedy Runtime: 0.067888 s  (95% CI: 0.058499, 0.077276)
Welfare Ratio (Greedy/VCG): 0.9998  (95% CI: 0.9986, 1.0000)
